In [17]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("../../../"))
PROJECT_ROOT = "../../../"

import lib.experimentation.experiments as exp
import lib.dimensionality_reduction.classical as dr
import lib.feature_selection.statistical_methods as stats_fs
import lib.learning_algorithms.classical as classical
import lib.learning_algorithms.domain_aware as domain_aware
import lib.learning_algorithms.ensemble as ensemble
import lib.learning_algorithms.predicate_based as predicate

# Load

In [18]:
data_path = os.path.join(
    PROJECT_ROOT,
    "Datasets/MotorImagery/processed/bci_features.csv"
)
df = exp.load_dataset(path=data_path)
df.head()

,subject,session,label,b8_12_mean_0,b8_12_mean_1,b8_12_mean_2,b8_12_mean_3,b8_12_mean_4,b8_12_mean_5,b8_12_mean_6,...,b13_30_logvar_12,b13_30_logvar_13,b13_30_logvar_14,b13_30_logvar_15,b13_30_logvar_16,b13_30_logvar_17,b13_30_logvar_18,b13_30_logvar_19,b13_30_logvar_20,b13_30_logvar_21
0,A01,session1,3,2.380351e-09,1.733224e-08,9.291380e-09,4.750105e-09,-4.489676e-09,-8.144126e-09,4.656935e-10,...,-25.187830,-25.044827,-24.994819,-24.918954,-24.897370,-24.826096,-25.067700,-24.979655,-24.907722,-24.815178
1,A01,session1,2,5.360754e-09,1.111883e-08,2.111726e-09,1.618129e-08,5.970768e-09,1.422088e-08,1.835344e-08,...,-25.250484,-25.262670,-25.195872,-25.144321,-25.104484,-24.889848,-25.114417,-25.048282,-25.019555,-24.819314
2,A01,session1,1,2.613426e-08,1.955509e-08,2.923330e-08,2.318268e-08,2.419033e-08,9.921166e-09,2.172376e-08,...,-25.189467,-25.272543,-25.242723,-25.123802,-25.139476,-25.155704,-25.211929,-25.092936,-25.112987,-24.934424
3,A01,session1,0,-2.354679e-10,-8.663889e-09,-7.688249e-09,-6.171389e-09,-1.202778e-09,2.668830e-09,-1.457442e-08,...,-25.413412,-25.065231,-25.032799,-24.997628,-25.140768,-25.254054,-24.937564,-24.935948,-25.031548,-24.746900
4,A01,session1,0,1.278455e-09,1.361218e-08,1.067621e-08,1.595427e-08,8.368821e-10,-2.031881e-09,-3.366320e-09,...,-25.160603,-25.293436,-25.238383,-25.097954,-25.181959,-25.163388,-25.133852,-25.051318,-25.071441,-24.730564


# Parameters

In [19]:
n_features = 64

fs_dict = {
    "mi": {
        "function": stats_fs.fs_mi,
        "params": {
            "n_features": n_features
        }
    }
}

In [ ]:
# ------------------------------------------------------------
# Predicate configs
# ------------------------------------------------------------
predicates_vapnik = [
    predicate.BiasPredicate(),
    predicate.LinearPredicate(),
]

predicates_gaussian = [
    predicate.GaussianFeatureClassPredicate()
]

predicates_mixed = [
    predicate.BiasPredicate(),
    predicate.LinearPredicate(),
    predicate.GaussianFeatureClassPredicate()
]

# ------------------------------------------------------------
# Model dict
# ------------------------------------------------------------
model_dict = {

    # --------------------------------------------------------
    # ERM baseline
    # --------------------------------------------------------
    "nn_erm": {
        "function": classical.train_nn_erm,
        "params": {
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 32
        }
    },
     "nn_domain_agreement": {
        "function": ensemble.train_domain_agreement,
        "params": {
            "model_kwargs": {
                "hidden_dim": 128,
                "output_dim": 4
            },
            "erm_epochs": 50,
            "align_epochs": 10,
            "sequential_rounds": 3,
            "lambda_agree": 0.1,
            "disagreement_mode": "consensus",
            "disagreement_distance": "l2",
            "do_alignment": True,
            "lr": 1e-3,
            "batch_size": 128
        }
    }
    # --------------------------------------------------------
    # Predicate NN (Vapnik style)
    # --------------------------------------------------------
    #"nn_pred_vapnik": {
    #    "function": predicate.train_predicate_nn_pipeline,
    #    "params": {
    #        "hidden_dim": 128,
    #        "epochs": 50,
    #        "lr": 1e-3,
    #        "batch_size": 32,
    #        "tau": 10,
    #        "predicates": predicates_gaussian
    #    }
    #}
}

# Experiments

In [21]:
global_result_df, global_time_df = exp.run_experiment(
    df,
    lambda x: exp.split_global_mixed_subjects(x, test_size=0.2, shuffle=True, random_state=42),
    fs_dict,
    model_dict
)
exp.save_results(global_result_df, global_time_df, "results_model/", "global")

Splits: 100%|██████████| 1/1 [00:19<00:00, 19.20s/it, type=global_mixed_subjects, subject=-, session=-]


In [22]:
intra_result_df, intra_time_df = exp.run_experiment(
    df, 
    lambda x: exp.split_intra_subject_session(x, test_size=0.2), 
    fs_dict, model_dict
    )
exp.save_results(intra_result_df, intra_time_df, "results_model/", "intra")

Splits: 100%|██████████| 18/18 [00:38<00:00,  2.14s/it, type=intra_session, subject=A09, session=session2]


In [23]:
inter_session_result_df, inter_session_time_df = exp.run_experiment(df, exp.split_inter_session, fs_dict, model_dict)
exp.save_results(inter_session_result_df, inter_session_time_df, "results_model/", "inter_session")

Splits: 100%|██████████| 18/18 [00:40<00:00,  2.23s/it, type=inter_session, subject=A09, session=session2]


In [24]:
inter_subject_result_df, inter_subject_time_df = exp.run_experiment(df, exp.split_inter_subject, fs_dict, model_dict)
exp.save_results(inter_subject_result_df, inter_subject_time_df, "results_model/", "inter_subject")

Splits: 100%|██████████| 9/9 [02:46<00:00, 18.48s/it, type=inter_subject, subject=A09, session=-]
